# B2-nolabel — Kaggle Test Census Execution (9-Hour Budget)

**Giao thức Đăng ký trước (OSF Frozen Protocol) — Bước 5/ Bước 7 Confirmatory Test:**
- **Arm:** `B2-nolabel` (B2-nolabel (Retrieved demo without label))
- **Tập dữ liệu:** Toàn bộ census Test (2,732 Devign + 2,274 ReVeal = 5,006 mẫu).
- **Ngưỡng khóa cố định (Locked Thresholds):** Devign $\tau^* = 0.98$, ReVeal $\tau^* = 0.21$.
  *(Không refit ngưỡng trên test; đánh giá trực tiếp theo quy tắc đã niêm phong)*.
- **Mô hình & Phần cứng:** `Qwen3-Coder-30B-A3B-Instruct` Q4_K_M GGUF trên 2x Tesla T4 (Kaggle Linux).
- **Cách ly Cache:** `repeat=False`.
- **Scoring:** Canonical first-token full-vocabulary logit scoring (`llm.eval` via `binary_scores`).


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone

SESSION_START_EPOCH = time.time()
SESSION_DEADLINE_EPOCH = SESSION_START_EPOCH + 9 * 3600
DIAG = Path('/kaggle/working/grace_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'

try:
    _gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                                check=True, capture_output=True, text=True)
    GPU_NAMES = [x.strip() for x in _gpu_probe.stdout.splitlines() if x.strip()]
except Exception as e:
    GPU_NAMES = []
print(f'Detected GPUs: {GPU_NAMES}', flush=True)

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    archive = shutil.make_archive('/kaggle/working/grace_diagnostics', 'zip', root_dir=DIAG)
    print('Tải log chẩn đoán từ Kaggle Output:', archive, flush=True)
    return archive

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Bắt đầu {name}: {cmd!r} | log: {log}')
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env)
        for line in proc.stdout:
            f.write(line); f.flush()
            print(line, end='', flush=True)
        code = proc.wait()
    note(f'{name} kết thúc với mã {code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

note('UTC: ' + datetime.now(timezone.utc).isoformat())
note('Python: ' + sys.version)
note('Platform: ' + platform.platform())


In [ ]:
import math
ARM = 'B2-nolabel'
ARM_SLUG = 'b2_nolabel'
IS_REPEAT = False
CACHE_TAG = ''
LOCKED_THRESHOLDS = {'devign': 0.98, 'reveal': 0.21}
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
BUNDLE_WORK = Path('/kaggle/working/b2_nolabel_test_bundles')
OUT_ROOT = Path('/kaggle/working/b2_nolabel_test')
BUNDLE_NAME = 'grace_b2_nolabel_test_inputs.zip'
BUNDLE_SHA256 = '8c6ca6cc022cb1e40afd35ff171d087c37605a006ffe8110d9961bf56cacf6a9'
CHECKPOINT_NAME = 'grace_b2_nolabel_test_checkpoint.zip'
CHECKPOINT_INPUT_PATH = None
MODEL_FILENAME = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_SHA256 = 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335'
MAX_RUNTIME_HOURS = 9
CHECKPOINT_RESERVE_SECONDS = 300
GPU_LAYERS = -1
SETS = [{"name": "seed_42/devign_test.zip", "sha256": "b0c8bd957d5906da3c4a3965421197056a5f4dd26f3828874b0b8018dcee4494", "seed": 42, "dataset": "devign", "prompts_sha256": "ee1afb588c429eb86921d5240f789967410ff0b05efc799564cc3b0a1eaa1fa6", "count": 2732, "tau_star": 0.98}, {"name": "seed_42/reveal_test.zip", "sha256": "2a29a6adb58b5cb482ec99fded3650b7a75369f0d92dad55a8166e3e80627d11", "seed": 42, "dataset": "reveal", "prompts_sha256": "f95aade82f8c7119fd32dac1017a968c79d14b8db3d655d37c87fe423f7a4ead", "count": 2274, "tau_star": 0.21}]


In [ ]:
# Extract and verify bundle inputs
def hash_bytes(data):
    return hashlib.sha256(data).hexdigest()

BUNDLE_WORK.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

archives = [p for p in KAGGLE_INPUT_ROOT.rglob(BUNDLE_NAME) if p.is_file()]
if archives:
    if hash_bytes(archives[0].read_bytes()) != BUNDLE_SHA256:
        raise ValueError('Bundle archive SHA-256 mismatch!')
    note(f'Bundle archive {BUNDLE_NAME} verified: SHA-256 matches.')
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(BUNDLE_WORK)
else:
    note(f'Bundle archive {BUNDLE_NAME} not found as single file; searching unpacked files.')

bundles = {}
for s in SETS:
    ds = s['dataset']
    inner_path = BUNDLE_WORK / s['name']
    if not inner_path.is_file():
        matches = [p for p in KAGGLE_INPUT_ROOT.rglob(Path(s['name']).name) if p.is_file()]
        if matches:
            inner_path = matches[0]
    if not inner_path.is_file():
        raise FileNotFoundError(f"Missing inner set: {s['name']}")
    if hash_bytes(inner_path.read_bytes()) != s['sha256']:
        raise ValueError(f"Inner bundle SHA-256 mismatch: {s['name']}")
    bundles[ds] = inner_path
    note(f'Set {ds} inner bundle verified: {inner_path}')

# Restore checkpoint from prior session if attached
if CHECKPOINT_INPUT_PATH is not None:
    checkpoint = Path(CHECKPOINT_INPUT_PATH)
else:
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob(CHECKPOINT_NAME))
    checkpoint = candidates[0] if candidates else None
if checkpoint and checkpoint.is_file():
    note(f'Khôi phục checkpoint từ: {checkpoint}')
    with zipfile.ZipFile(checkpoint) as z:
        for name in z.namelist():
            if name.endswith('records.jsonl') or name.endswith('run_manifest.json'):
                target = OUT_ROOT / name
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(z.read(name))
    note('Khôi phục checkpoint thành công.')


In [ ]:
# Cài đặt exact packages và compile llama-cpp-python CUDA
note('Installing pinned packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                'transformers==4.57.1', 'huggingface_hub==0.36.2', 'tokenizers==0.22.2',
                'scikit-learn==1.5.2', 'numpy==2.1.3'], check=True)

env = os.environ.copy()
env['CMAKE_ARGS'] = '-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64'
env['CMAKE_BUILD_PARALLEL_LEVEL'] = '2'
env['FORCE_CMAKE'] = '1'
note('Compiling llama-cpp-python==0.3.16 with CUDA SM75...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-binary',
                'llama-cpp-python', 'llama-cpp-python==0.3.16'], env=env, check=True)

import llama_cpp
note(f'Installed llama_cpp version: {llama_cpp.__version__}')


In [ ]:
# Tìm model GGUF và kiểm tra SHA-256
model_candidates = sorted(KAGGLE_INPUT_ROOT.rglob(MODEL_FILENAME))
if not model_candidates:
    raise FileNotFoundError(f'Model {MODEL_FILENAME} not found in /kaggle/input')
MODEL_PATH = model_candidates[0]
note(f'Found model: {MODEL_PATH}')

h = hashlib.sha256()
with open(MODEL_PATH, 'rb') as f:
    for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''):
        h.update(chunk)
actual_model_sha = h.hexdigest()
if actual_model_sha != MODEL_SHA256:
    raise ValueError(f'Model SHA-256 mismatch! Expected {MODEL_SHA256}, got {actual_model_sha}')
note('Model SHA-256 verified successfully!')


In [ ]:
%%writefile /kaggle/working/kaggle_test_inference.py
import argparse, hashlib, json, math, os, platform, time, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import llama_cpp
from transformers import AutoTokenizer

def binary_scores(logits, label_ids):
    values = np.asarray(logits, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(values)):
        raise ValueError('Non-finite logits')
    maximum = np.max(values)
    normalizer = float(maximum + np.log(np.exp(values - maximum).sum()))
    lp0 = float(values[label_ids[0]] - normalizer)
    lp1 = float(values[label_ids[1]] - normalizer)
    score = float(np.exp(lp1 - np.logaddexp(lp0, lp1)))
    best = int(np.argmax(values))
    top = np.argsort(values)[-5:][::-1]
    return {'logprob_0': lp0, 'logprob_1': lp1, 'p0': math.exp(lp0), 'p1': math.exp(lp1),
            's': score, 'pred_argmax': int(score >= 0.5), 'greedy_token_id': best,
            'top5': [{'token_id': int(i), 'logprob': float(values[i] - normalizer)} for i in top]}

def run_inference(arm, model_path, bundle_zip, out_dir, tau_star, is_repeat=False, cache_tag=None):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rec_path = out_dir / 'records.jsonl'
    man_path = out_dir / 'run_manifest.json'
    eval_path = out_dir / 'evaluation.json'
    
    with zipfile.ZipFile(bundle_zip) as z:
        manifest = json.loads(z.read('bundle_manifest.json'))
        prompts = [json.loads(line) for line in z.read('prompts.jsonl').decode('utf-8').splitlines() if line.strip()]
    
    ds = manifest['dataset']
    total_count = len(prompts)
    print(f'Starting inference for {arm} {ds} test: {total_count} prompts, fixed tau={tau_star}')
    
    done_ids = set()
    if rec_path.is_file():
        for line in rec_path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                row = json.loads(line)
                done_ids.add(str(row['id']))
        print(f'Resuming: already completed {len(done_ids)}/{total_count}')
    
    tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-Coder-30B-A3B-Instruct', revision='b2cff646eb4bb1d68355c01b18ae02e7cf42d120')
    llm = llama_cpp.Llama(
        model_path=str(model_path),
        n_ctx=8192,
        n_batch=256,
        n_ubatch=256,
        n_gpu_layers=-1,
        split_mode=llama_cpp.LLAMA_SPLIT_MODE_LAYER,
        seed=4294967295,
        logits_all=True,
        verbose=False
    )
    
    label_ids = []
    for label in ('0', '1'):
        hf_ids = tok.encode(label, add_special_tokens=False)
        gguf_ids = llm.tokenize(label.encode(), add_bos=False, special=False)
        if len(hf_ids) != 1 or hf_ids != gguf_ids:
            raise ValueError(f'Label token mismatch for {label}: HF={hf_ids}, GGUF={gguf_ids}')
        label_ids.append(hf_ids[0])
    if label_ids[0] == label_ids[1]:
        raise ValueError('Label token IDs must differ')
    
    arm_slug = arm.lower().replace('-', '_')
    checkpoint_file = Path('/kaggle/working') / f'grace_{arm_slug}_test_checkpoint.zip'
    out_root = out_dir.parent.parent.parent
    
    def save_snapshot():
        tmp_zip = checkpoint_file.with_suffix('.zip.tmp')
        with zipfile.ZipFile(tmp_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
            for p in out_root.rglob('*'):
                if p.is_file() and not p.name.endswith('.tmp'):
                    z.write(p, p.relative_to(out_root).as_posix())
        os.replace(tmp_zip, checkpoint_file)
    
    last_snapshot_time = time.time()
    
    with open(rec_path, 'a', encoding='utf-8') as out_f:
        for idx, item in enumerate(prompts):
            uid = str(item['id'])
            if uid in done_ids:
                continue
            
            p_text = item['prompt']
            messages = [{'role': 'user', 'content': p_text}]
            chat = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            hf_ids = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=False)
            gguf_ids = llm.tokenize(chat.encode('utf-8'), add_bos=False, special=True)
            if list(hf_ids) != gguf_ids:
                raise ValueError(f'Tokenizer mismatch at {ds}/{uid}; abort rather than alter prompt')
            if len(gguf_ids) > 8192:
                raise ValueError(f'Prompt exceeded context budget: {len(gguf_ids)} > 8192')
            
            start = time.monotonic()
            llm.reset()
            llm.eval(gguf_ids)
            last_logits = llm.scores[llm.n_tokens - 1]
            res = binary_scores(last_logits, label_ids)
            elapsed_s = time.monotonic() - start
            
            raw = llm.detokenize([res['greedy_token_id']]).decode('utf-8', errors='replace').strip()
            raw_valid = raw in ('0', '1')
            s_val = res['s']
            pred_tau = int(s_val >= tau_star)
            
            rec = {
                'id': uid,
                'arm': arm,
                'seed': 42,
                'dataset': ds,
                'split': 'test',
                'label': item['label'],
                's': s_val,
                'valid': 1,
                'pred_argmax': res['pred_argmax'],
                'tau': tau_star,
                'pred_tau': pred_tau,
                'raw': raw,
                'response_valid': raw_valid,
                'logprob_0': res['logprob_0'],
                'logprob_1': res['logprob_1'],
                'p0': res['p0'],
                'p1': res['p1'],
                'greedy_token_id': res['greedy_token_id'],
                'top5': res['top5'],
                'token_ids': {'0': label_ids[0], '1': label_ids[1]},
                'prompt_sha256': item['prompt_sha256'],
                'model_sha256': 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335',
                'inference_seed': 4294967295,
                'elapsed_s': elapsed_s,
                'timestamp': datetime.now(timezone.utc).isoformat(),
            }
            if is_repeat:
                rec['repeat'] = True
                rec['tag'] = cache_tag or 'repeat'
            
            out_f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            out_f.flush()
            os.fsync(out_f.fileno())
            done_ids.add(uid)
            
            now = time.time()
            print(f'{arm}/{ds} {uid}: s={s_val:.4f}, raw={raw!r} ({len(done_ids)}/{total_count})', flush=True)
            
            if len(done_ids) % 100 == 0 or (now - last_snapshot_time) >= 120:
                save_snapshot()
                last_snapshot_time = now
    
    save_snapshot()
    
    # Compute final test metrics at fixed tau*
    from sklearn.metrics import matthews_corrcoef, roc_auc_score, balanced_accuracy_score, f1_score
    all_recs = [json.loads(l) for l in rec_path.read_text(encoding='utf-8').splitlines() if l.strip()]
    y_true = np.array([r['label'] for r in all_recs])
    y_pred = np.array([r['pred_tau'] for r in all_recs])
    y_score = np.array([r['s'] for r in all_recs])
    
    eval_summary = {
        'arm': arm,
        'dataset': ds,
        'split': 'test',
        'seed': 42,
        'tau_locked': tau_star,
        'n_total': len(all_recs),
        'n_valid': sum(r['valid'] for r in all_recs),
        'mcc_at_tau': float(matthews_corrcoef(y_true, y_pred)),
        'auroc': float(roc_auc_score(y_true, y_score)),
        'ba_at_tau': float(balanced_accuracy_score(y_true, y_pred)),
        'f1_at_tau': float(f1_score(y_true, y_pred, zero_division=0)),
        'completed_at_utc': datetime.now(timezone.utc).isoformat()
    }
    eval_path.write_text(json.dumps(eval_summary, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'Finished {arm} {ds} test: MCC={eval_summary["mcc_at_tau"]:.4f}, AUROC={eval_summary["auroc"]:.4f}')
    
    run_manifest = {
        'arm': arm,
        'dataset': ds,
        'split': 'test',
        'seed': 42,
        'count': len(all_recs),
        'bundle_sha256': manifest.get('bundle_sha256'),
        'prompts_sha256': manifest.get('prompts_sha256'),
        'records_sha256': hashlib.sha256(rec_path.read_bytes()).hexdigest(),
        'model_sha256': 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335',
        'backend': f'llama-cpp-python=={llama_cpp.__version__}',
        'quantization': 'Q4_K_M',
        'model_repo': 'ggml-org/Qwen3-Coder-30B-A3B-Instruct-GGUF',
        'tokenizer': 'Qwen/Qwen3-Coder-30B-A3B-Instruct',
        'tokenizer_revision': 'b2cff646eb4bb1d68355c01b18ae02e7cf42d120',
        'inference_seed': 4294967295,
        'tau_star': tau_star,
        'evaluation': eval_summary
    }
    if is_repeat:
        run_manifest['repeat'] = True
        run_manifest['tag'] = cache_tag or 'repeat'
    man_path.write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding='utf-8')

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--arm', required=True)
    parser.add_argument('--model', required=True)
    parser.add_argument('--bundle', required=True)
    parser.add_argument('--out', required=True)
    parser.add_argument('--tau', type=float, required=True)
    parser.add_argument('--repeat', action='store_true')
    parser.add_argument('--tag', default='repeat')
    args = parser.parse_args()
    run_inference(args.arm, args.model, args.bundle, args.out, args.tau, args.repeat, args.tag)


In [ ]:
# Chạy suy luận lần lượt cho Devign và ReVeal test
for ds in ['devign', 'reveal']:
    tau = LOCKED_THRESHOLDS[ds]
    inner_bundle = bundles[ds]
    out_dir = OUT_ROOT / 'seed_42' / ds / 'test'
    note(f'Bắt đầu test inference {ARM} {ds} (tau={tau})...')
    cmd = [sys.executable, '/kaggle/working/kaggle_test_inference.py',
           '--arm', ARM, '--model', str(MODEL_PATH), '--bundle', str(inner_bundle),
           '--out', str(out_dir), '--tau', str(tau)]
    if IS_REPEAT:
        cmd += ['--repeat', '--tag', CACHE_TAG]
    run_logged(cmd, f'test_{ARM_SLUG}_{ds}')
    
    # Refresh downloadable checkpoint
    shutil.make_archive('/kaggle/working/grace_b2_nolabel_test_checkpoint', 'zip', root_dir=OUT_ROOT)
    note(f'Đã cập nhật checkpoint sau {ds}!')

# Export final output zip
final_zip = shutil.make_archive('/kaggle/working/grace_b2_nolabel_test_final_output', 'zip', root_dir=OUT_ROOT)
note(f'HOÀN TẤT TOÀN BỘ TEST CENSUS: {final_zip}')
diagnostics_zip()
